# Notebook 01 - 从零理解 PyResOps 分层

这个 Notebook 面向**刚接触项目**和**水库知识还不系统**的同学。

学习目标：
- 建立最小水库调度直觉（入流、出流、库容、水位）
- 看懂 `tools/services/domain/core/constraints/rules/metrics/modules/plugins/storage` 10 层分别做什么
- 先不纠结细节，先知道整条链路在解决什么业务问题


## 1) 水库调度最小模型（先有业务直觉）

你可以把水库当成一个可控水桶：
- 入流 `Q_in`：上游来水
- 出流 `Q_out`：我们可控制的下泄
- 库容 `S`：桶里有多少水
- 水位 `H`：库容映射出来的液位高度

核心约束是水量平衡：

`S(t+1) = S(t) + (Q_in - Q_out) * dt`

PyResOps 的本质就是：在每个时间步决定一个合理的 `Q_out`，让防洪/供水/生态/发电等目标尽量都被满足。


In [ ]:
def water_balance_step(storage_yi_m3, inflow_m3s, outflow_m3s, dt_seconds=3600):
    """
    storage 单位: 亿m^3
    inflow/outflow 单位: m^3/s
    """
    delta_storage = (inflow_m3s - outflow_m3s) * dt_seconds / 1e8
    return storage_yi_m3 + delta_storage

storage = 20.0
inflows = [6000, 8000, 10000, 9000, 7000, 5000]
outflows = [6500, 7000, 7500, 8000, 7500, 7000]

print(f"初始库容: {storage:.3f} 亿m^3")
for i, (qin, qout) in enumerate(zip(inflows, outflows), start=1):
    storage = water_balance_step(storage, qin, qout)
    print(f"第{i}小时: 入流={qin:>5} 出流={qout:>5} -> 库容={storage:.3f} 亿m^3")


## 2) 10 层目录各自的设计作用

把它类比成一家餐厅：
- `tools`：前台接单（MCP 工具）
- `services`：店长编排流程（调用谁、先后顺序）
- `domain`：菜单和订单的标准数据结构（统一语言）
- `core`：后厨核心烹饪逻辑（仿真、规则决策、约束校核）
- `constraints`：硬性规则（不能违反的红线）
- `rules`：策略规则（在什么条件下采取什么动作）
- `metrics`：评分体系（这次做得好不好）
- `modules`：具体操作手法（恒定下泄、蓄水量驱动等）
- `plugins`：扩展机制（以后加新约束/规则/指标）
- `storage`：账本和档案（SQLite 持久化）


In [ ]:
from pathlib import Path

root = Path('pyresops')
layers = [
    'tools', 'services', 'domain', 'core', 'constraints',
    'rules', 'metrics', 'modules', 'plugins', 'storage'
]

for layer in layers:
    layer_dir = root / layer
    py_files = sorted(p.name for p in layer_dir.glob('*.py'))
    print(f"{layer:12} -> {len(py_files)} files")
    if py_files:
        print('   ', ', '.join(py_files[:4]), '...' if len(py_files) > 4 else '')


## 3) 串起来最终在做什么

最终目标不是“跑代码”，而是形成**可解释、可复盘、可比较**的调度决策：

1. 接收当前快照和未来来水预报
2. 生成/选择一个调度方案（模块序列 + 切换条件）
3. 仿真逐步推进，得到每小时状态轨迹
4. 用约束判断是否违规
5. 用指标体系打分
6. 记录决策轨迹并持久化
7. 在滚动场景下持续重评估/替换/定版


In [ ]:
pipeline = [
    'snapshot + forecast',
    'program generation',
    'simulation (core)',
    'constraint check',
    'metric evaluation',
    'decision trace + storage',
    'rolling reassessment/finalization'
]

for i, step in enumerate(pipeline, start=1):
    print(f"{i}. {step}")


## 4) 下一步

继续看 `notebook_02_domain_modules_core.ipynb`，我们会真正创建 `ReservoirSpec/DispatchProgram` 并跑一段最小仿真，让分层不只是概念。
